## Metadata API batch generator

This notebook fetches metadata source records from `https://api.data.niaid.nih.gov/v1/metadata` and maps each `src.*.sourceInfo` block into an NDE `ResourceCatalog` JSON record.

Mapped properties:
- `name`
- `description`
- `abstract`
- `genre`
- `conditionsOfAccess`
- `url`
- `identifier`
- `dataset`

Output:
- a dated batch file in `../nde-metadata-corrections/metadata_for_DDE/dataCatalogs/`
- optional individual JSON files for each source in the same directory


In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import requests


In [ ]:
script_path = Path.cwd()
parent_path = script_path.parent
result_path = parent_path / 'nde-metadata-corrections' / 'metadata_for_DDE' / 'dataCatalogs'
metadata_url = 'https://api.data.niaid.nih.gov/v1/metadata'
write_individual_files = True
request_timeout = 60

context_dict = {
    'owl': 'http://www.w3.org/2002/07/owl#',
    'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
    'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
    'schema': 'http://schema.org/',
    'niaid': 'https://discovery.biothings.io/ns/niaid/',
    'nde': 'https://discovery.biothings.io/ns/nde/'
}

included_in_catalog = {
    '@type': 'DataCatalog',
    'name': 'Data Discovery Engine',
    'url': 'https://discovery.biothings.io/'
}

print(f'Results will be written to: {result_path}')


In [ ]:
def fetch_metadata(url: str) -> dict:
    response = requests.get(url, timeout=request_timeout)
    response.raise_for_status()
    return response.json()


def clean_nones(value):
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            if item in [None, '', [], {}, 'None']:
                continue
            cleaned[key] = clean_nones(item)
        return cleaned
    if isinstance(value, list):
        cleaned = [clean_nones(item) for item in value]
        return [item for item in cleaned if item not in [None, '', [], {}, 'None']]
    return value


def safe_filename(value: str) -> str:
    filename = re.sub(r'[\\/:*?"<>|]+', '_', value).strip()
    return filename or 'data_catalog'


def build_dataset_object(name: str | None) -> dict | None:
    if not name:
        return None
    return {
        'name': f'{name} datasets ingested into the NIAID Data Ecosystem Discovery Portal',
        'url': 'https://data.niaid.nih.gov/search?filters=%28date%3A%5B%222000-01-01%22+TO+%222026-12-31%22%5D+OR+%28-_exists_%3A%28%22date%22%29%29%29+AND+%28includedInDataCatalog.name%3A%28%22' + name + '%22%29%29&from=1'
    }


def normalize_to_list(value):
    if isinstance(value, list):
        return value
    if value in [None, '', [], {}, 'None']:
        return []
    return [value]


In [ ]:
def build_resource_catalog(source_payload: dict, context: dict) -> dict | None:
    source_info = source_payload.get('sourceInfo') or {}
    if not source_info:
        return None

    name = source_info.get('name')
    record = {
        '@type': 'nde:ResourceCatalog',
        '@context': context,
        'date': datetime.now().strftime('%Y-%m-%d'),
        'includedInDataCatalog': included_in_catalog,
        'name': name,
        'description': source_info.get('description'),
        'abstract': source_info.get('abstract'),
        'genre': source_info.get('genre'),
        'conditionsOfAccess': source_info.get('conditionsOfAccess'),
        'url': source_info.get('url'),
        'identifier': source_info.get('identifier'),
        'dataset': build_dataset_object(name)
    }
    return clean_nones(record)


def process_records(payload: dict, context: dict) -> list[dict]:
    batchlist = []
    source_map = payload.get('src', {})
    for source_payload in source_map.values():
        record = build_resource_catalog(source_payload, context)
        if record is not None:
            batchlist.append(record)
    batchlist.sort(key=lambda item: item.get('name', '').lower())
    return batchlist


In [ ]:
payload = fetch_metadata(metadata_url)
batchlist = process_records(payload, context_dict)

print(f'Fetched {len(batchlist)} ResourceCatalog records')
print(batchlist[0])


In [ ]:
result_path.mkdir(parents=True, exist_ok=True)
today = datetime.now().strftime('%Y-%m-%d')
batch_filepath = result_path / f'{today}_api_batch_file.json'

with batch_filepath.open('w', encoding='utf-8') as outfile:
    json.dump(batchlist, outfile, indent=4, ensure_ascii=False)

print(f'Wrote batch file: {batch_filepath}')


In [ ]:
if write_individual_files:
    for record in batchlist:
        filename_root = safe_filename(record.get('identifier') or record.get('name', 'data_catalog'))
        output_filepath = result_path / f'{filename_root}.json'
        with output_filepath.open('w', encoding='utf-8') as outfile:
            json.dump(record, outfile, indent=4, ensure_ascii=False)
    print(f'Wrote {len(batchlist)} individual data catalog files')
else:
    print('Skipping individual file generation')


In [ ]:
required_fields = ['name', 'description', 'abstract', 'genre', 'conditionsOfAccess', 'url', 'identifier', 'dataset']
summary = {
    field: sum(1 for record in batchlist if field in record)
    for field in required_fields
}
summary
